# Task 2: Unsupervised Domain Adaptation (PACS)

Source-only → DAN → DANN → CDAN on Sketch, plus λ_MMD study and final evaluation.

Checkpoint selection uses **mean source-val macro-F1 only** (no Sketch labels).

In [ ]:
import torch

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

REPO_DIR = Path("/content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA1")
os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

In [ ]:
%pip install -q -r requirements.txt gdown

## Download PACS (once)

Standard DomainBed-style zip. After unzip, domain folders must live at `data/pacs/{photo,art_painting,cartoon,sketch}/`.

In [ ]:
from pathlib import Path
import shutil
import zipfile
import urllib.request

pacs_root = Path("data/pacs")
marker = pacs_root / "photo"

def _count(d):
    return sum(1 for p in (pacs_root / d).rglob("*") if p.is_file())

if marker.is_dir() and _count("photo") > 0:
    print("PACS already present at", pacs_root.resolve())
else:
    pacs_root.parent.mkdir(parents=True, exist_ok=True)
    zip_path = Path("data/PACS.zip")
    # Prefer HuggingFace zip (DomainBed Google Drive id is often rate-limited).
    if not zip_path.exists():
        url = "https://huggingface.co/datasets/Azeez577/PACS/resolve/main/PACS.zip"
        print("Downloading", url)
        urllib.request.urlretrieve(url, zip_path)
    extract_dir = Path("data/_pacs_extract")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    candidates = [c.parent for c in extract_dir.rglob("photo") if c.is_dir()]
    if not candidates:
        raise FileNotFoundError("Could not locate PACS photo/ after unzip")
    src = candidates[0]
    if pacs_root.exists():
        shutil.rmtree(pacs_root)
    shutil.move(str(src), str(pacs_root))
    print("Installed PACS ->", pacs_root.resolve())

for d in ["photo", "art_painting", "cartoon", "sketch"]:
    print(f"  {d}: {_count(d)} files")

In [ ]:
!python -m shared.prepare_pacs_splits

## Train main methods

Order: Source-only (also Task 3 ERM baseline) → DAN → DANN → CDAN.

In [ ]:
!python -m task2.scripts.run_task2 --stages train_main

## Controlled study (DAN λ_MMD ∈ {0.1, 1, 10})

In [ ]:
!python -m task2.scripts.run_task2 --stages study_lambda

## Final evaluation (Sketch labels allowed only here)

In [ ]:
!python -m task2.scripts.run_task2 --stages eval

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path("task2/results/tables/task2_main_comparison.json").read_text())
for row in summary["table"]:
    print(
        f"{row['run']:24s}  srcF1={row['mean_source_macro_f1']:.3f}  "
        f"sketchAcc={row['target_accuracy']:.3f}  "
        f"dSep={row['domain_separability']:.3f}  "
        f"dAcc={row.get('target_accuracy_change_vs_source_only')}"
    )